# AI-Driven Audit & Compliance Validator
## TCS AMD Hackathon 2026

This notebook demonstrates the full audit pipeline:
1. **Document Intake** — PDF parsing with PyMuPDF + pdfplumber (OCR optional)
2. **Two-RAG System** — ChromaDB with bge-large-en-v1.5 embeddings
3. **103 Built-in Rules** — Comprehensive SEBI ICDR/LODR/SAST compliance rules
4. **5-Layer Validation** — Deterministic + NLP + Semantic + Numerical + Cross-Reference
5. **Self-Reflection Critic** — Reviews findings, catches false positives
6. **Confidence Scoring** — Multi-signal weighted scores
7. **Pipeline Metrics** — End-to-end latency, GPU usage, token tracking
8. **Audit Report** — PDF export with full audit trail

---


## Step 0: Install Dependencies
Run this cell once to install all required packages.


In [ ]:
# Install dependencies (run once)
!pip install -q openai sentence-transformers chromadb \
    langgraph langchain-core langchain-openai \
    PyMuPDF pdfplumber Pillow \
    fpdf2 pandas scikit-learn tqdm \
    plotly psutil


## Step 1: Configuration

**IMPORTANT:** Adjust the paths below to match your environment:
- `AUDIT_BASE_DIR` — Directory containing `compliance_guidelines/` and `ipo_documents/`
- `AUDIT_SHARED_DIR` — Persistent storage directory (survives notebook restart)
- `VLLM_BASE_URL` — Your vLLM server endpoint


In [ ]:
import os
import sys
import time

# ============================================================
# CONFIGURE THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================

# AMD Cloud (notebook.amd.com / Jupyter Lab)
os.environ['AUDIT_BASE_DIR'] = 'Amd-Tcs-hackathon'
os.environ['AUDIT_SHARED_DIR'] = 'shared'
os.environ['VLLM_BASE_URL'] = 'http://localhost:8000/v1'

# OCR is disabled by default (PDFs have selectable text)
os.environ['USE_OCR'] = 'false'

# Add project root to Python path
project_root = os.environ['AUDIT_BASE_DIR']
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Also add workspace root if needed
workspace = '/workspace/Amd-Tcs-hackathon'
if os.path.isdir(workspace) and workspace not in sys.path:
    sys.path.insert(0, workspace)

print(f'Project root: {project_root}')
print(f'Shared dir:   {os.environ["AUDIT_SHARED_DIR"]}')
print(f'vLLM URL:     {os.environ["VLLM_BASE_URL"]}')
print(f'OCR enabled:  {os.environ["USE_OCR"]}')


In [ ]:
# Verify configuration and directory structure
from src.config import ensure_dirs, print_config, COMPLIANCE_DIR, IPO_DIR

ensure_dirs()
print_config()

# Verify data exists
compliance_files = os.listdir(COMPLIANCE_DIR) if os.path.isdir(COMPLIANCE_DIR) else []
ipo_files = [f for f in os.listdir(IPO_DIR) if f.endswith('.pdf')] if os.path.isdir(IPO_DIR) else []

print(f'\nCompliance PDFs: {len(compliance_files)} files')
for f in compliance_files:
    print(f'  - {f}')

print(f'\nIPO Documents: {len(ipo_files)} PDFs')
print(f'  First 5: {ipo_files[:5]}')


## Step 2: Start vLLM Server

**Run this in a separate terminal** (not in this notebook):

```bash
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-72B-Instruct \
    --dtype bfloat16 \
    --tensor-parallel-size 1 \
    --port 8000 \
    --max-model-len 16384 \
    --gpu-memory-utilization 0.90 \
    --trust-remote-code
```

Wait for the server to show `Uvicorn running on http://0.0.0.0:8000` before proceeding.


In [ ]:
# Test LLM connection
from src.llm_client import LLMClient

llm = LLMClient()

# Health check
print('Testing LLM connection...')
is_healthy = llm.health_check()
if is_healthy:
    print('✅ LLM server is running and responsive!')
else:
    print('❌ ERROR: LLM server is not responding.')
    print('Make sure vLLM is running on the correct port.')
    print(f'Trying to reach: {os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")}')


## Step 3: Initialize Embedding Model & ChromaDB


In [ ]:
from src.embeddings import EmbeddingManager

# This will download bge-large-en-v1.5 (~1.3GB) on first run
# Set device='cpu' if GPU should be reserved for vLLM only
emb_manager = EmbeddingManager(
    device='cuda'  # Change to 'cpu' if GPU memory is tight
)

print(f'\nExisting collections: {emb_manager.list_collections()}')


## Step 4: Index Compliance Documents (One-time Setup)

This parses the 3 SEBI regulation PDFs, chunks them by regulation number, 
and indexes them into ChromaDB. Only needs to run once — results persist in `shared/chroma_db/`.


In [ ]:
from src.agent import AuditAgent

# Initialize the agent
agent = AuditAgent(llm_client=llm, embedding_manager=emb_manager)

# Index compliance documents (skip if already done)
compliance_chunks = agent.setup_compliance_rag(force_reindex=False)


## Step 5: Load Compliance Rules

We ship **103 built-in SEBI compliance rules** covering:
- **ICDR Schedule VI** — Cover page, risk factors, capital structure, objects, basis for price, tax, industry, business, management, promoters, dividends, financials, legal info
- **LODR Corporate Governance** — Board composition, committees, whistle blower, insider trading
- **SAST** — Public shareholding, promoter holding


In [ ]:
# Load built-in 103 rules (instant)
from src.rule_generator import DEFAULT_SEBI_RULES, print_rules_summary

rules = DEFAULT_SEBI_RULES
print(f'Loaded {len(rules)} built-in compliance rules')

# Count by type and severity
by_type = {}
by_sev = {}
for r in rules:
    by_type[r['check_type']] = by_type.get(r['check_type'], 0) + 1
    by_sev[r['severity']] = by_sev.get(r['severity'], 0) + 1
print(f'  By type:     {by_type}')
print(f'  By severity: {by_sev}')


## Step 6: Initialize Pipeline Metrics

We track per-step timing, GPU usage, memory, and LLM token consumption.


In [ ]:
from src.metrics import PipelineMetrics

# Create metrics tracker — this snapshots GPU & system info at init time
metrics = PipelineMetrics()
print(f'GPU detected: {metrics.gpu_info}')
print(f'System info:  {metrics.system_info}')


## Step 7: Test Document Parsing

Let's test the document parser on a sample DRHP before running the full audit.


In [ ]:
from src.document_parser import parse_document
from src.config import IPO_DIR

# Safety check: Ensure PDFs exist
if not os.path.isdir(IPO_DIR) or not any(f.endswith('.pdf') for f in os.listdir(IPO_DIR)):
    raise FileNotFoundError(
        f"No PDF files found in '{IPO_DIR}'.\n\n"
        f"Make sure to run the Configuration cell (Step 1) to set the correct paths."
    )

# Pick a sample DRHP to test
sample_files = sorted([f for f in os.listdir(IPO_DIR) if f.endswith('.pdf')])[:5]
print('Available sample DRHPs:')
for i, f in enumerate(sample_files):
    size_mb = os.path.getsize(os.path.join(IPO_DIR, f)) / (1024*1024)
    print(f'  {i}: {f} ({size_mb:.1f} MB)')

# Choose the first one (change index as needed)
SAMPLE_IDX = 0
sample_pdf = os.path.join(IPO_DIR, sample_files[SAMPLE_IDX])
print(f'\nSelected: {sample_files[SAMPLE_IDX]}')


## Step 8: Run Full Audit (with Pipeline Metrics)

This runs the complete pipeline with timing instrumentation:
1. Parse document (LLM-powered ToC extraction)
2. Index into ChromaDB
3. Run all 103 validation checks (deterministic + semantic + cross-reference)
4. Critic reviews findings
5. Compute confidence scores
6. Generate report

**Expected time: 15-25 minutes** with 103 rules.


In [ ]:
# ── Run the full audit pipeline with metrics tracking ──
import time as _time

audit_start = _time.time()

# Step 8a: Parse
metrics.start_step('document_parsing')
audit_state = agent.run_audit(
    document_path=sample_pdf,
    rules=rules
)
total_time = _time.time() - audit_start

# The agent runs all steps internally, so we record overall timing
metrics.end_step('document_parsing', extra={'pages': audit_state.parsed_doc.total_pages})

# We can estimate step breakdown from the audit trail
print(f'\n⏱️  Total audit time: {total_time:.1f}s ({total_time/60:.1f} min)')


## Step 9: View Results


In [ ]:
# Display findings table
from src.report_generator import print_findings_table

print_findings_table(audit_state.verified_findings)


In [ ]:
# View detailed findings for non-compliant items
import json

non_compliant = [f for f in audit_state.verified_findings if f['status'] != 'COMPLIANT']

print(f'\n{"="*70}')
print(f'  NON-COMPLIANT AND NEEDS-REVIEW FINDINGS ({len(non_compliant)})')
print(f'{"="*70}\n')

for i, finding in enumerate(non_compliant, 1):
    status_icon = {'NON_COMPLIANT': 'FAIL', 'NEEDS_REVIEW': 'REVIEW'}.get(finding['status'], '?')
    print(f'--- Finding {i}: [{status_icon}] {finding["rule_title"]} ---')
    print(f'  Rule ID:     {finding["rule_id"]}')
    print(f'  Regulation:  {finding["regulation_ref"]}')
    print(f'  Severity:    {finding["severity"]}')
    print(f'  Confidence:  {finding["confidence"]:.0%}')
    print(f'  Check Type:  {finding["check_type"]}')
    print(f'  Explanation: {finding.get("explanation", "N/A")}')
    print(f'  Evidence:    {finding.get("evidence", {}).get("document_excerpt", "N/A")[:200]}')
    if finding.get('critic_reasoning'):
        print(f'  Critic Note: {finding["critic_reasoning"]}')
    print()


In [ ]:
# View false positives caught by critic
if audit_state.discarded_findings:
    print(f'\nFalse Positives Caught by Critic Agent ({len(audit_state.discarded_findings)}):')
    print(f'{"-"*60}')
    for d in audit_state.discarded_findings:
        print(f'  Rule: {d["rule_id"]} - {d["rule_title"]}')
        print(f'  Reason: {d.get("discard_reason", "N/A")}')
        print()
else:
    print('\nNo false positives were caught by the critic agent.')


In [ ]:
# View score summary
print(f'\n{"="*50}')
print(f'  COMPLIANCE SCORE SUMMARY')
print(f'{"="*50}')
summary = audit_state.score_summary
print(f'  Overall Score:       {audit_state.overall_score:.0%}')
print(f'  Total Rules Checked: {summary.get("total_rules_checked", 0)}')
print(f'  Compliant:           {summary.get("compliant", 0)}')
print(f'  Non-Compliant:       {summary.get("non_compliant", 0)}')
print(f'  Needs Review:        {summary.get("needs_review", 0)}')
print(f'  Avg Confidence:      {summary.get("average_confidence", 0):.0%}')
print(f'\n  By Severity: {json.dumps(summary.get("by_severity", {}), indent=4)}')
print(f'  By Check Type: {summary.get("by_check_type", {})}')


## Step 10: Cache Results for Streamlit Dashboard

Save the audit results + metrics so the Streamlit dashboard can load them instantly.


In [ ]:
from cache_results import cache_audit_state

# Build metrics dict
llm_stats = llm.get_stats()
doc_stats = {
    'total_pages': audit_state.parsed_doc.total_pages,
    'tables_extracted': audit_state.parsed_doc.tables_count,
    'sections_detected': len(audit_state.parsed_doc.sections),
    'ocr_pages': audit_state.parsed_doc.ocr_pages_count,
}
metrics_dict = metrics.to_dict(llm_stats=llm_stats, doc_stats=doc_stats)

# Cache the results
cache_path = cache_audit_state(
    state=audit_state,
    metrics=metrics_dict,
    output_dir=os.environ.get('AUDIT_SHARED_DIR', 'shared')
)
print(f'\n✅ Cached to: {cache_path}')
print(f'   Streamlit can now load this instantly!')


## Step 11: LLM Usage Statistics


In [ ]:
# Print usage stats (now with accurate token tracking)
stats = llm.get_stats()
print(f'\n{"="*50}')
print(f'  LLM USAGE STATISTICS')
print(f'{"="*50}')
print(f'  Total API calls:    {stats["total_calls"]}')
print(f'  Total tokens:       {stats["total_tokens"]:,}')
print(f'  ├─ Input tokens:    {stats["total_input_tokens"]:,}')
print(f'  └─ Output tokens:   {stats["total_output_tokens"]:,}')
print(f'  Avg tokens/call:    {stats["avg_tokens_per_call"]:,}')
print(f'  Avg latency/call:   {stats["avg_latency_sec"]:.1f}s')
print(f'  Total LLM time:     {stats["total_llm_time_sec"]:.0f}s ({stats["total_llm_time_sec"]/60:.1f} min)')
print(f'  Wall clock time:    {stats["wall_clock_sec"]:.0f}s ({stats["wall_clock_sec"]/60:.1f} min)')

critic_stats = agent.critic.get_stats()
print(f'\n  CRITIC STATISTICS')
print(f'  Valid findings:      {critic_stats["valid"]}')
print(f'  False positives:     {critic_stats["false_positive"]}')
print(f'  Needs more evidence: {critic_stats["needs_more"]}')
print(f'  Errors:              {critic_stats["errors"]}')


## Step 12: Pipeline Performance Metrics


In [ ]:
# Display comprehensive metrics
print(f'\n{"="*50}')
print(f'  PIPELINE PERFORMANCE METRICS')
print(f'{"="*50}')
print(f'  Total duration:     {metrics.get_total_duration():.0f}s ({metrics.get_total_duration()/60:.1f} min)')
print(f'\n  GPU:')
print(f'    Available:        {metrics.gpu_info["available"]}')
print(f'    Name:             {metrics.gpu_info["name"]}')
print(f'    VRAM Total:       {metrics.gpu_info["memory_total_gb"]} GB')
print(f'    VRAM Used:        {metrics.gpu_info["memory_used_gb"]} GB')
print(f'\n  System:')
print(f'    RAM Total:        {metrics.system_info["ram_total_gb"]} GB')
print(f'    RAM Used:         {metrics.system_info["ram_used_gb"]} GB')
print(f'    CPU Cores:        {metrics.system_info["cpu_count"]}')

# Current memory snapshot
snap = metrics.snapshot_memory()
print(f'\n  Current Memory:')
print(f'    RAM Used:         {snap.get("ram_used_gb", "N/A")} GB ({snap.get("ram_pct", "N/A")}%)')
if 'gpu_used_gb' in snap:
    print(f'    GPU VRAM Used:    {snap["gpu_used_gb"]} GB')


## Step 13: Generate PDF Report


In [ ]:
from src.report_generator import generate_pdf_report
from src.agent import save_report

# Save JSON report
reports_dir = os.path.join(os.environ['AUDIT_BASE_DIR'], 'reports')
json_path = save_report(audit_state, output_dir=reports_dir)

# Generate PDF report
pdf_path = json_path.replace('.json', '.pdf')
generate_pdf_report(audit_state.report, pdf_path)

print(f'\nReports saved to: {reports_dir}')
print(f'  JSON: {json_path}')
print(f'  PDF:  {pdf_path}')


## Step 14: Run on Second Document

Audit a second document and cache those results too.


In [ ]:
# Pick another DRHP to audit
ipo_files = sorted([f for f in os.listdir(IPO_DIR) if f.endswith('.pdf')])

print(f'Available documents ({len(ipo_files)}):')
for i, f in enumerate(ipo_files[:20]):
    size_mb = os.path.getsize(os.path.join(IPO_DIR, f)) / (1024*1024)
    print(f'  {i:3d}: {f} ({size_mb:.1f} MB)')


In [ ]:
# Run audit on second document
DOCUMENT_INDEX = 1  # Change this to pick a different file

next_pdf = os.path.join(IPO_DIR, ipo_files[DOCUMENT_INDEX])
print(f'Auditing: {ipo_files[DOCUMENT_INDEX]}')

# Reset metrics for second run
metrics2 = PipelineMetrics()
metrics2.start_step('full_audit')

state2 = agent.run_audit(document_path=next_pdf, rules=rules)

metrics2.end_step('full_audit')

# Show results
print_findings_table(state2.verified_findings)

# Cache for Streamlit
llm_stats2 = llm.get_stats()
doc_stats2 = {
    'total_pages': state2.parsed_doc.total_pages,
    'tables_extracted': state2.parsed_doc.tables_count,
    'sections_detected': len(state2.parsed_doc.sections),
    'ocr_pages': state2.parsed_doc.ocr_pages_count,
}
metrics_dict2 = metrics2.to_dict(llm_stats=llm_stats2, doc_stats=doc_stats2)

cache_path2 = cache_audit_state(
    state=state2,
    metrics=metrics_dict2,
    output_dir=os.environ.get('AUDIT_SHARED_DIR', 'shared')
)
print(f'\n✅ Cached to: {cache_path2}')

# Save report
save_report(state2, output_dir=reports_dir)


## Step 15: Comparison Summary


In [ ]:
# Side-by-side comparison
print(f'\n{"="*70}')
print(f'  COMPARISON: TWO DOCUMENT AUDITS')
print(f'{"="*70}')

for i, (state, label) in enumerate([(audit_state, 'Document 1'), (state2, 'Document 2')]):
    s = state.score_summary
    company = state.parsed_doc.metadata.get('company_name', 'Unknown')
    print(f'\n  {label}: {company}')
    print(f'    Score:         {state.overall_score:.0%}')
    print(f'    Compliant:     {s.get("compliant", 0)}')
    print(f'    Non-Compliant: {s.get("non_compliant", 0)}')
    print(f'    Needs Review:  {s.get("needs_review", 0)}')
    print(f'    FP Caught:     {len(state.discarded_findings)}')
    print(f'    Pages:         {state.parsed_doc.total_pages}')
    print(f'    Tables:        {state.parsed_doc.tables_count}')

final_stats = llm.get_stats()
print(f'\n  TOTAL LLM USAGE (both documents):')
print(f'    API calls:     {final_stats["total_calls"]}')
print(f'    Tokens:        {final_stats["total_tokens"]:,}')
print(f'    LLM time:      {final_stats["total_llm_time_sec"]:.0f}s')


## Step 16: Launch Streamlit Dashboard

Run this in a **terminal** (not in the notebook):

```bash
pip install streamlit plotly psutil
streamlit run app.py --server.port 8501 --server.address 0.0.0.0
```

The dashboard will load the cached audit results instantly (no 19-minute wait).

---

## Summary

| Feature | Description |
|---|---|
| **103 Built-in Rules** | Comprehensive SEBI ICDR/LODR/SAST compliance rules |
| **5-Layer Validation** | Deterministic + NLP + Semantic + Numerical + Cross-Reference |
| **Self-Reflection** | Critic agent catches false positives |
| **Retry Retrieval** | 3-strategy retrieval reduces NEEDS_REVIEW rate |
| **Confidence Scoring** | Multi-signal weighted formula (not arbitrary numbers) |
| **Full Auditability** | Every step logged with timestamps |
| **Pipeline Metrics** | End-to-end latency, GPU/memory, token tracking |
| **Streamlit Dashboard** | Interactive results visualization with Plotly |
